# ⚠️ Churn Prediction — XGBoost + SHAP

**Phase 6** of the Customer Intelligence Platform.

This notebook:

1. Engineers per-customer behavioural features from transactions.
2. Labels customers as **churned** (no purchase in the last 90 days) or **active**.
3. Trains an **XGBoost** binary classifier with class-weight balancing.
4. Evaluates with AUC-ROC / AUC-PR.
5. Explains predictions with **SHAP** values.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import preprocess
from src.churn_model import build_churn_features, train_churn_model, save_model
from src.visualizations import set_plot_style, save_fig

set_plot_style()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("✅ Setup complete")

## 1. Load & clean transactions

In [ ]:
df = preprocess(keep_returns=False, save=False)
print(f"Cleaned transactions: {len(df):,} rows")

## 2. Build churn features

Engineer per-customer features and label churn (no purchase in > 90 days).

In [ ]:
features = build_churn_features(df, churn_days=90)
print("\nFeature columns:", list(features.drop(columns="churn").columns))
features.describe()

In [ ]:
# Churn class balance
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=features["churn"], ax=ax, palette=["#55A868", "#C44E52"])
ax.set_title("Churn class distribution")
ax.set_xticklabels(["Active", "Churned"])
ax.set_xlabel("")
save_fig(fig, "churn_class_balance", subdir="churn")
plt.show()

## 3. Train XGBoost

In [ ]:
model, metrics, shap_df = train_churn_model(features)

## 4. Feature importance (gain-based)

In [ ]:
importance = pd.Series(model.feature_importances_, index=model.feature_names_in_).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importance.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("XGBoost feature importance (gain)")
ax.set_xlabel("Importance")
save_fig(fig, "churn_feature_importance", subdir="churn")
plt.show()

## 5. SHAP explanations

SHAP values tell us **why** each customer was predicted as churned or active.

In [ ]:
if not shap_df.empty:
    import shap as shap_lib

    # Summary plot — global feature impact
    plt.figure()
    shap_lib.summary_plot(shap_df.values, features=shap_df.columns, show=False)
    plt.title("SHAP summary — churn prediction")
    plt.tight_layout()
    save_fig(plt.gcf(), "shap_summary", subdir="churn")
    plt.show()
else:
    print("⚠️ SHAP values not available (shap package not installed)")

In [ ]:
if not shap_df.empty:
    # SHAP waterfall for the top-3 most-likely churners
    test_features = features.drop(columns="churn")
    y_prob = model.predict_proba(test_features.loc[shap_df.index])[:, 1]
    top_churners = shap_df.index[y_prob.argsort()[-3:]]

    for cid in top_churners:
        plt.figure(figsize=(8, 4))
        shap_lib.waterfall_plot(
            shap_lib.Explanation(
                values=shap_df.loc[cid].values,
                base_values=model.predict_proba(test_features.iloc[[0]])[:, 1].mean(),
                data=test_features.loc[cid].values,
                feature_names=test_features.columns.tolist(),
            ),
            show=False,
        )
        plt.title(f"SHAP waterfall — Customer {cid} (P(churn)={y_prob[shap_df.index.get_loc(cid)]:.2f})")
        plt.tight_layout()
        save_fig(plt.gcf(), f"shap_waterfall_{cid}", subdir="churn")
        plt.show()
else:
    print("⚠️ SHAP waterfall plots skipped")

## 6. Save model for dashboard

In [ ]:
save_model(model)

## 7. Summary & next steps

### Key takeaways
- **Recency** and **frequency** dominate churn prediction — the longer a customer is inactive, the more likely they are to churn.
- **SHAP** provides per-customer explanations — every prediction is actionable.
- The model is saved to ``models/churn_model.json`` for the dashboard.

### Next step
→ **Phase 7**: Multi-page Streamlit dashboard integrating all three layers (RFM + LTV + Churn).

---
© 2026 AdamAI-Systems.